In [1]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn import datasets
from sklearn.tree import DecisionTreeClassifier
import numpy as np
from Testers import Tester
import pandas as pd
from Bagging import get_accuracy, create_bags, create_models

In [2]:
data = datasets.load_digits()
X = data.data
y = data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [3]:
from Testers import test_method
from BaggingGEN import BaggingGEN

trees_n = [5, 10, 15]

reps = 2

def test_gen(n_trees):
    gen = BaggingGEN(X=X_train, y=y_train, n_trees=n_trees,
                     max_iterations=200, crossover_rate=0.7, mutation_rate=0.05)
    models =  gen.run_genetic_algorithm()
    fitness = get_accuracy(X_test, y_test, models)
    return fitness


def test_bagging(n_trees):
    bags = create_bags(X=X_train, y=y_train, n_bags=n_trees, with_replacement=False)
    models = create_models(bags=bags, n_trees=n_trees)
    fitness = get_accuracy(X_test, y_test, models)
    return fitness
    
    


for n in trees_n:
    test_method(test_gen(n), args=None, save_path="./src/accuracy_bagging_gen_{n}.csv", reps=reps)
    test_method(test_bagging(n), args=None, save_path="./src/accuracy_bagging_{n}.csv", reps=reps)




IndexError: index 50 is out of bounds for axis 0 with size 8

In [ ]:
# Create plot
import matplotlib.pyplot as plt

def get_accuracy_mean_std(name: str, trees_n:int):
    df = pd.read_csv(f'./../res/accuracy_{name}_{trees_n}.csv')
    accuracy = df['Accuracy']
    return accuracy.mean(), accuracy.std()

X = trees_n

Y1 = [get_accuracy_mean_std('bagging', i)[0] for i in X]
Y2 = [get_accuracy_mean_std('bagging_gen', i)[0] for i in X]

Y1_std = [get_accuracy_mean_std('bagging', i)[1] for i in X]
Y2_std = [get_accuracy_mean_std('bagging_sa', i)[1] for i in X]

plt.figure(figsize=(10, 6))
plt.plot(X, Y1, marker='o', label='Bagging')
plt.plot(X, Y2, marker='o', label='Bagging SA')

plt.fill_between(X, np.array(Y1) - np.array(Y1_std), np.array(Y1) + np.array(Y1_std), alpha=0.2)
plt.fill_between(X, np.array(Y2) - np.array(Y2_std), np.array(Y2) + np.array(Y2_std), alpha=0.2)

plt.title('Accuracy vs Number of Trees (mean ± std)')
plt.xlabel('Number of Trees')
plt.ylabel('Accuracy')
plt.xticks(X)
plt.grid()
plt.legend()
plt.tight_layout()
plt.savefig('./../res/accuracy_vs_n_trees.png')
plt.show()

# save to csv
compared_by_accuracy = pd.DataFrame({
    'Trees_n': X,
    'Bagging': Y1,
    'bagging_std': Y1_std,
    'Bagging SA': Y2,
    'bagging_sa_std': Y2_std,
})
compared_by_accuracy.to_csv('./../res/accuracy_vs_n_trees.csv', index=False)
    